In [42]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras import Model

MODEL_PATH = './MobileNetV3_ASL_AdamW_phase1.h5'

try:
    # Try to load full model (architecture + weights)
    model = tf.keras.models.load_model(MODEL_PATH, safe_mode=False)
    print("Loaded full model from HDF5.")
except Exception as e:
    print("load_model failed, attempting weights-only fallback:", e)
    # Build backbone
    base = MobileNetV3Large(input_shape=(224,224,3), include_top=False, weights=None)
    x = GlobalAveragePooling2D()(base.output)
    out = Dense(2, activation="softmax")(x)  # minimal head
    model = Model(inputs=base.input, outputs=out)
    try:
        model.load_weights(MODEL_PATH, by_name=True)
        print("Loaded weights by_name into backbone.")
    except Exception as e2:
        print("Failed to load weights by_name:", e2)

model.summary()


load_model failed, attempting weights-only fallback: Only input tensors may be passed as positional arguments. The following argument value should be passed as a keyword argument: 3.0 (of type <class 'float'>)
Loaded weights by_name into backbone.


Model: "functional_36"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_27      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_27        │ (None, 224, 224,  │          0 │ input_layer_27[0… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv (Conv2D)       │ (None, 112, 112,  │        432 │ rescaling_27[0][… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_bn             │ (None, 112, 112,  │         64 │ conv[0][0]        │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_540      │ (None, 112, 112,  │          0 │ conv_bn[0][0]     │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        144 │ activation_540[0… │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │         64 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_513 (ReLU)    │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        256 │ re_lu_513[0][0]   │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_add   │ (None, 112, 112,  │          0 │ activation_540[0… │
│ (Add)               │ 16)               │            │ expanded_conv_pr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_ex… │ (None, 112, 112,  │      1,024 │ expanded_conv_ad… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_ex… │ (None, 112, 112,  │        256 │ expanded_conv_1_… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_514 (ReLU)    │ (None, 112, 112,  │          0 │ expanded_conv_1_… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, 113, 113,  │          0 │ re_lu_514[0][0]   │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, 56, 56,    │        576 │ expanded_conv_1_… │
│ (DepthwiseConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, 56, 56,    │        256 │ expanded_conv_1_

 Total params: 2,998,274 (11.44 MB)

 Trainable params: 2,973,874 (11.34 MB)

 Non-trainable params: 24,400 (95.31 KB)

In [46]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras import Model

# ==========================================
# 1. Configuration & Paths
# ==========================================
INPUT_DIR = './isic-2024'
OUTPUT_DIR = './gradecam_result'
MODEL_PATH = './MobileNetV3_ASL_AdamW_phase1.h5'

# Fixed dictionary/list syntax error here
CLASS_NAMES = ["benign", "malignant"] # Index 0: Non-cancer, Index 1: Cancer

# Assigned to the last 4D convolutional layer of the model
LAST_CONV_LAYER_NAME = 'conv_1' 


# ==========================================
# 2. Core Functions
# ==========================================
def build_grad_model(model, layer_name):
    """
    Constructs a gradient model that outputs BOTH the activation of the 
    last conv layer and the final model predictions.
    """
    try:
        grad_model = tf.keras.models.Model(
            inputs=[model.inputs],
            outputs=[model.get_layer(layer_name).output, model.output]
        )
        return grad_model
    except ValueError as e:
        print(f"Error: Layer '{layer_name}' not found in the model.")
        print("Please check model.summary() and update LAST_CONV_LAYER_NAME.")
        raise e


def gradcam_plus(img_batch, grad_model):
    """
    Upgraded Grad-CAM++ implementation.
    Returns: (heatmap, predictions)
    """
    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(img_batch, training=False)
        # Target the class with the highest prediction score
        loss = preds[:, tf.argmax(preds[0])]

    # Get gradients of the loss with respect to the feature maps
    grads = tape.gradient(loss, conv_outputs)

    # Approximate 2nd and 3rd derivatives
    g2, g3 = grads ** 2, grads ** 3

    sum_activations = tf.reduce_sum(conv_outputs, axis=(1, 2), keepdims=True)
    alpha = g2 / (2.0 * g2 + sum_activations * g3 + 1e-8)

    # Calculate channel weights
    weights = tf.reduce_sum(alpha * tf.nn.relu(grads), axis=(1, 2))

    # Weight the feature maps and sum across channels
    heatmap = tf.reduce_sum(conv_outputs[0] * weights, axis=-1)

    # Apply ReLU (keep only positive values)
    heatmap = tf.maximum(heatmap, 0.0)

    # Normalize between 0 and 1
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    # Return both the processed heatmap and the original softmax predictions
    return (heatmap ** 2.5).numpy(), preds.numpy()


def save_gradcam_overlay(img_path, heatmap, output_path, label, confidence, alpha=0.4):
    """Applies the heatmap over the original image, writes the prediction, and saves it."""
    # Load original image
    img = cv2.imread(img_path)
    
    # Resize heatmap to match original image dimensions
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    
    # Convert heatmap to RGB
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    # Superimpose the heatmap on original image
    superimposed_img = heatmap * alpha + img
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)
    
    # Write the prediction text on the image
    text = f"{label}: {confidence:.1f}%"
    # Choose color based on prediction (Green for benign, Red for malignant)
    text_color = (0, 255, 0) if label == "benign" else (0, 0, 255) 
    cv2.putText(superimposed_img, text, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 1, text_color, 2, cv2.LINE_AA)
    
    # Save the result
    cv2.imwrite(output_path, superimposed_img)


# ==========================================
# 3. Main Execution Block
# ==========================================
def main():
    # Ensure the output directory exists
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Loading base model...")
    try:
        # Try to load full model (architecture + weights)
        base_model = tf.keras.models.load_model(MODEL_PATH, safe_mode=False)
        print("Loaded full model from HDF5.")
    except Exception as e:
        print(f"load_model failed: {e}\nAttempting weights-only fallback...")
        # Build backbone fallback for MobileNetV3
        base = MobileNetV3Large(input_shape=(224,224,3), include_top=False, weights=None)
        x = GlobalAveragePooling2D()(base.output)
        out = Dense(2, activation="softmax")(x)
        base_model = Model(inputs=base.input, outputs=out)
        
        try:
            base_model.load_weights(MODEL_PATH, by_name=True)
            print("Loaded weights by_name into backbone.")
        except Exception as e2:
            print(f"Failed to load weights: {e2}")
            return
    
    print(f"Building Grad-CAM model using layer: '{LAST_CONV_LAYER_NAME}'...")
    grad_model = build_grad_model(base_model, LAST_CONV_LAYER_NAME)

    print(f"Processing images from '{INPUT_DIR}'...")
    
    # Iterate through the input folder
    for filename in os.listdir(INPUT_DIR):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            
            input_file_path = os.path.join(INPUT_DIR, filename)
            output_file_path = os.path.join(OUTPUT_DIR, f"cam_{filename}")
            
            # Load and preprocess image
            img = tf.keras.preprocessing.image.load_img(input_file_path, target_size=(224, 224))
            img_array = tf.keras.preprocessing.image.img_to_array(img)
            img_batch = np.expand_dims(img_array, axis=0)
            
            # MobileNetV3 standard preprocessing
            img_batch = tf.keras.applications.mobilenet_v3.preprocess_input(img_batch)
            
            # Generate Heatmap AND extract predictions
            heatmap, preds = gradcam_plus(img_batch, grad_model=grad_model) 
            
            # Identify Cancer vs Non-Cancer using Softmax Output
            pred_idx = np.argmax(preds[0]) # Get index of highest probability (0 or 1)
            confidence = preds[0][pred_idx] * 100 # Convert to percentage
            predicted_label = CLASS_NAMES[pred_idx]
            
            # Save the result with the label overlaid
            save_gradcam_overlay(input_file_path, heatmap, output_file_path, predicted_label, confidence)
            
            # Console logging
            print(f"Processed: {filename} | Prediction: {predicted_label} ({confidence:.2f}%)")

    print("\nAll Grad-CAM++ visualizations completed successfully.")

# Run the script
if __name__ == "__main__":
    main()

Loading base model...
load_model failed: Only input tensors may be passed as positional arguments. The following argument value should be passed as a keyword argument: 3.0 (of type <class 'float'>)
Attempting weights-only fallback...
Loaded weights by_name into backbone.
Building Grad-CAM model using layer: 'conv_1'...
Processing images from './isic-2024'...


c:\Users\prome\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\models\functional.py:258: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: [['keras_tensor_8174']]
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


Processed: clean_ISIC_0052109.jpg | Prediction: malignant (58.56%)
Processed: clean_ISIC_0119495.jpg | Prediction: benign (57.44%)
Processed: clean_ISIC_0157834.jpg | Prediction: benign (57.45%)
Processed: clean_ISIC_0190307.jpg | Prediction: benign (53.26%)
Processed: clean_ISIC_0211092.jpg | Prediction: malignant (59.59%)

All Grad-CAM++ visualizations completed successfully.
